<a href="https://colab.research.google.com/github/VietorMoyo/NMT_model/blob/main/nmt.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q datasets pandas pyarrow tqdm

In [2]:
from datasets import load_dataset, Audio

REPO_ID = "Teleagents/zimbabwe-all-speech"
SUBSETS = ["ndebele_openbible", "shona_bible"]

for subset in SUBSETS:
    print(f"\n{'='*20} {subset} {'='*20}")
    ds = load_dataset(REPO_ID, subset, split="train", streaming=True)

    # Disable audio decoding so no audio arrays are pulled into memory
    if "audio" in ds.features:
        ds = ds.cast_column("audio", Audio(decode=False))

    sample = next(iter(ds))
    print("Available columns:", list(sample.keys()))
    for k, v in sample.items():
        if k == "audio" and isinstance(v, dict):
            print(f"  audio.path: {v.get('path')}")
        else:
            print(f"  {k}: {repr(v)[:120]}")


==================== ndebele_openbible ====================


README.md:   0%|          | 0.00/12.3k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/35 [00:00<?, ?it/s]

Available columns: ['audio', 'text', 'testament', 'book', 'chapter', 'verse', 'duration_seconds', 'speaker_id']
  audio.path: MAL_001_Verse_003.flac
  text: 'kodwa u-Esawu ngimzondile, futhi ngenze izintaba zakhe zaba lugwadule lelifa lakhe ladliwa ngamakhanka asenkangala.”'
  testament: 'Old Testament'
  book: 'Malachi'
  chapter: '001'
  verse: '003'
  duration_seconds: 10.0
  speaker_id: 'SPEAKER_00'

==================== shona_bible ====================


Available columns: ['audio', 'transcription', 'speaker_id', 'speaker_sex', 'language', 'book', 'chapter', 'verse', 'verse_part', 'verse_part_count', 'duration']
  audio.path: Genesis_chapter_001_verse_001.flac
  transcription: 'Pakutanga Mwari akasika matenga nenyika.'
  speaker_id: 1
  speaker_sex: 'male'
  language: 'sna'
  book: 'Genesis'
  chapter: 1
  verse: 1
  verse_part: 1
  verse_part_count: 1
  duration: 3.369999885559082


In [3]:
import os
import re
import json
import pandas as pd
from pathlib import Path
from datasets import load_dataset, Audio

REPO_ID = "Teleagents/zimbabwe-all-speech"
OUT_DIR = Path("data")
OUT_DIR.mkdir(exist_ok=True)

def clean_text(text: str) -> str:
    if not isinstance(text, str):
        return ""
    return re.sub(r"\s+", " ", text).strip()

def normalize_book(book_name: str) -> str:
    """Normalizes book names (e.g. '1 John', 'First John', 'GENESIS' -> 'genesis')."""
    if not book_name:
        return ""
    b = str(book_name).strip().lower()
    # Replace Roman numerals or standard abbreviations if any exist
    b = re.sub(r"[^\w\s]", "", b)
    return re.sub(r"\s+", "_", b)

# ==========================================
# 1. Load & Process Ndebele OpenBible
# ==========================================
print("Loading and preparing Ndebele OpenBible...")
ds_nd = load_dataset(REPO_ID, "ndebele_openbible", split="train")
ds_nd = ds_nd.cast_column("audio", Audio(decode=False))

nd_records = []
for row in ds_nd:
    try:
        book = normalize_book(row["book"])
        chapter = int(str(row["chapter"]).strip())
        verse = int(str(row["verse"]).strip())
        verse_key = f"{book}_{chapter}_{verse}"
        text = clean_text(row["text"])

        if len(text) > 2:
            nd_records.append({
                "verse_key": verse_key,
                "text_nd": text
            })
    except (ValueError, TypeError):
        continue

df_nd = pd.DataFrame(nd_records)
# Deduplicate / merge any duplicate rows
df_nd = df_nd.groupby("verse_key", as_index=False).agg({"text_nd": " ".join})
print(f"Processed {len(df_nd)} unique Ndebele verses.")

# ==========================================
# 2. Load & Process Shona Bible
# ==========================================
print("\nLoading and preparing Shona Bible...")
ds_sn = load_dataset(REPO_ID, "shona_bible", split="train")
ds_sn = ds_sn.cast_column("audio", Audio(decode=False))

sn_records = []
for row in ds_sn:
    try:
        book = normalize_book(row["book"])
        chapter = int(str(row["chapter"]).strip())
        verse = int(str(row["verse"]).strip())
        verse_part = int(row.get("verse_part", 1) or 1)
        verse_key = f"{book}_{chapter}_{verse}"
        text = clean_text(row["transcription"])

        if len(text) > 2:
            sn_records.append({
                "verse_key": verse_key,
                "verse_part": verse_part,
                "text_sn": text
            })
    except (ValueError, TypeError):
        continue

df_sn = pd.DataFrame(sn_records)
# Sort by verse_part before grouping so split verses are combined in the right order
df_sn = df_sn.sort_values(by=["verse_key", "verse_part"])
df_sn = df_sn.groupby("verse_key", as_index=False).agg({"text_sn": " ".join})
print(f"Processed {len(df_sn)} unique Shona verses.")

# ==========================================
# 3. Inner Join Parallel Verses
# ==========================================
aligned = pd.merge(df_sn, df_nd, on="verse_key", how="inner")
print(f"\n==========================================")
print(f"Successfully aligned {len(aligned)} parallel Shona-Ndebele verses!")
print(f"==========================================")

if len(aligned) == 0:
    print("\n[!] 0 matches. Inspecting sample keys:")
    print("  Shona sample keys:  ", df_sn["verse_key"].head(5).tolist())
    print("  Ndebele sample keys:", df_nd["verse_key"].head(5).tolist())
else:
    # 4. Create bidirectional pairs (<2nd> for Ndebele target, <2sn> for Shona target)
    pairs = []
    for _, row in aligned.iterrows():
        # Shona -> Ndebele
        pairs.append({
            "verse_key": row["verse_key"],
            "src": f"<2nd> {row['text_sn']}",
            "tgt": row["text_nd"]
        })
        # Ndebele -> Shona
        pairs.append({
            "verse_key": row["verse_key"],
            "src": f"<2sn> {row['text_nd']}",
            "tgt": row["text_sn"]
        })

    # 5. Shuffle and split 95% train / 5% validation
    pairs_df = pd.DataFrame(pairs).sample(frac=1.0, random_state=42).reset_index(drop=True)
    split_idx = int(len(pairs_df) * 0.95)

    train_df = pairs_df.iloc[:split_idx]
    val_df = pairs_df.iloc[split_idx:]

    train_df.to_json(OUT_DIR / "train.json", orient="records", force_ascii=False, indent=2)
    val_df.to_json(OUT_DIR / "val.json", orient="records", force_ascii=False, indent=2)

    # 6. Save raw bilingual corpus for BPE Tokenizer training
    with open(OUT_DIR / "corpus.txt", "w", encoding="utf-8") as f:
        for _, row in aligned.iterrows():
            f.write(row["text_sn"] + "\n")
            f.write(row["text_nd"] + "\n")

    print(f"Saved: {len(train_df)} training pairs to {OUT_DIR / 'train.json'}")
    print(f"Saved: {len(val_df)} validation pairs to {OUT_DIR / 'val.json'}")
    print(f"Saved: Combined text corpus to {OUT_DIR / 'corpus.txt'}")

Loading and preparing Ndebele OpenBible...


Resolving data files:   0%|          | 0/35 [00:00<?, ?it/s]

data/ndebele_openbible/train/train-00000(…): reconstructing file:   0%|          |  0.00B /  376MB            

data/ndebele_openbible/train/train-00000(…): downloading bytes:           |  0.00B            

data/ndebele_openbible/train/train-00001(…): reconstructing file:   0%|          |  0.00B /  373MB            

...

data/shona_bible/validation/validation-0(…): downloading bytes:           |  0.00B            

Generating test split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Processed 27122 unique Shona verses.

Successfully aligned 26301 parallel Shona-Ndebele verses!
Saved: 49971 training pairs to data/train.json
Saved: 2631 validation pairs to data/val.json
Saved: Combined text corpus to data/corpus.txt


In [4]:
# Option A: Mount Google Drive and copy files
from google.colab import drive
import shutil

drive.mount('/content/drive')
drive_dest = Path('/content/drive/MyDrive/nmt_bible_data')
drive_dest.mkdir(exist_ok=True)

for fname in ["train.json", "val.json", "corpus.txt"]:
    shutil.copy(f"data/{fname}", drive_dest / fname)
print(f"Persisted dataset files to {drive_dest}")

# Option B: Download directly to your local machine (uncomment if preferred)
# from google.colab import files
# files.download('data/train.json')
# files.download('data/val.json')
# files.download('data/corpus.txt')

Mounted at /content/drive
Persisted dataset files to /content/drive/MyDrive/nmt_bible_data


In [5]:
!pip install -q sentencepiece torch

In [6]:
import sentencepiece as spm
from pathlib import Path

MODEL_DIR = Path("tokenizer")
MODEL_DIR.mkdir(exist_ok=True)
MODEL_PREFIX = str(MODEL_DIR / "sn_nd_bpe")

# 1. Train shared Shona-Ndebele BPE tokenizer (8,000 subword tokens)
spm.SentencePieceTrainer.train(
    input="data/corpus.txt",
    model_prefix=MODEL_PREFIX,
    vocab_size=8000,
    model_type="bpe",
    character_coverage=1.0,
    pad_id=0,
    bos_id=1,
    eos_id=2,
    unk_id=3,
    user_defined_symbols=["<2nd>", "<2sn>"]
)

# 2. Load the trained tokenizer and verify special token IDs
sp = spm.SentencePieceProcessor(model_file=f"{MODEL_PREFIX}.model")

print(f"Total Vocab Size: {sp.get_piece_size()}")
print(f"Special IDs -> PAD: {sp.pad_id()}, BOS: {sp.bos_id()}, EOS: {sp.eos_id()}, UNK: {sp.unk_id()}")
print(f"Direction IDs -> <2nd>: {sp.piece_to_id('<2nd>')}, <2sn>: {sp.piece_to_id('<2sn>')}\n")

# 3. Test subword segmentation on a sample Shona & Ndebele sentence
sample_sn = "<2nd> Pakutanga Mwari akasika matenga nenyika."
sample_nd = "<2sn> Ekuqaleni uNkulunkulu wadala amazulu lomhlaba."

for text in [sample_sn, sample_nd]:
    pieces = sp.encode(text, out_type=str)
    ids = sp.encode(text, out_type=int)
    decoded = sp.decode(ids)
    print(f"Original: {text}")
    print(f"Subwords: {pieces}")
    print(f"Token IDs: {ids}")
    print(f"Decoded:  {decoded}\n")

Total Vocab Size: 8000
Special IDs -> PAD: 0, BOS: 1, EOS: 2, UNK: 3
Direction IDs -> <2nd>: 4, <2sn>: 5

Original: <2nd> Pakutanga Mwari akasika matenga nenyika.
Subwords: ['▁', '<2nd>', '▁Paku', 'tanga', '▁Mwari', '▁aka', 'sika', '▁matenga', '▁nenyika', '.']
Token IDs: [7902, 4, 4337, 808, 311, 80, 892, 5212, 3203, 7924]
Decoded:  <2nd> Pakutanga Mwari akasika matenga nenyika.

Original: <2sn> Ekuqaleni uNkulunkulu wadala amazulu lomhlaba.
Subwords: ['▁', '<2sn>', '▁Eku', 'qaleni', '▁uNkulunkulu', '▁wa', 'dala', '▁amazulu', '▁lomhlaba', '.']
Token IDs: [7902, 5, 2604, 5637, 417, 63, 1335, 6523, 4167, 7924]
Decoded:  <2sn> Ekuqaleni uNkulunkulu wadala amazulu lomhlaba.



In [7]:
import json
import torch
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

PAD_IDX = sp.pad_id()  # 0
BOS_IDX = sp.bos_id()  # 1
EOS_IDX = sp.eos_id()  # 2

class BibleTranslationDataset(Dataset):
    def __init__(self, json_path: str, tokenizer: spm.SentencePieceProcessor, max_len: int = 128):
        with open(json_path, "r", encoding="utf-8") as f:
            self.pairs = json.load(f)
        self.sp = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        pair = self.pairs[idx]
        # Encode source and target, truncating to max_len - 2 to leave room for BOS and EOS
        src_ids = self.sp.encode(pair["src"], out_type=int)[:self.max_len - 2]
        tgt_ids = self.sp.encode(pair["tgt"], out_type=int)[:self.max_len - 2]

        src_tensor = torch.tensor([BOS_IDX] + src_ids + [EOS_IDX], dtype=torch.long)
        tgt_tensor = torch.tensor([BOS_IDX] + tgt_ids + [EOS_IDX], dtype=torch.long)
        return src_tensor, tgt_tensor

def collate_batch(batch):
    src_list, tgt_list = zip(*batch)
    # Pad sequences in the current batch with PAD_IDX (0)
    src_padded = pad_sequence(src_list, batch_first=True, padding_value=PAD_IDX)
    tgt_padded = pad_sequence(tgt_list, batch_first=True, padding_value=PAD_IDX)
    return src_padded, tgt_padded

# Create Datasets and DataLoaders
BATCH_SIZE = 64
train_dataset = BibleTranslationDataset("data/train.json", sp)
val_dataset = BibleTranslationDataset("data/val.json", sp)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_batch)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_batch)

# Test fetching a single batch
src_batch, tgt_batch = next(iter(train_loader))
print(f"Train batches: {len(train_loader)} | Val batches: {len(val_loader)}")
print(f"Sample src_batch shape: {src_batch.shape} (batch_size, src_seq_len)")
print(f"Sample tgt_batch shape: {tgt_batch.shape} (batch_size, tgt_seq_len)")

Train batches: 781 | Val batches: 42
Sample src_batch shape: torch.Size([64, 61]) (batch_size, src_seq_len)
Sample tgt_batch shape: torch.Size([64, 65]) (batch_size, tgt_seq_len)


In [8]:
import random
import torch
import torch.nn as nn

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

# 1. Bidirectional GRU Encoder
class Encoder(nn.Module):
    def __init__(self, vocab_size, embed_dim, enc_hidden_dim, dec_hidden_dim, dropout=0.25):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=PAD_IDX)
        self.rnn = nn.GRU(embed_dim, enc_hidden_dim, bidirectional=True, batch_first=True)
        # Linear layer to compress 2*enc_hidden_dim -> dec_hidden_dim for initial decoder state
        self.fc = nn.Linear(enc_hidden_dim * 2, dec_hidden_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, src):
        # src: [batch_size, src_len]
        embedded = self.dropout(self.embedding(src))
        enc_outputs, hidden = self.rnn(embedded)
        # enc_outputs: [batch_size, src_len, enc_hidden_dim * 2]
        # hidden: [2, batch_size, enc_hidden_dim] (forward + backward)

        # Concat last forward & backward hidden states and project to decoder dimension
        dec_init_hidden = torch.tanh(self.fc(torch.cat((hidden[-2], hidden[-1]), dim=1)))
        return enc_outputs, dec_init_hidden

# 2. Bahdanau Additive Attention
class Attention(nn.Module):
    def __init__(self, enc_hidden_dim, dec_hidden_dim):
        super().__init__()
        self.attn = nn.Linear((enc_hidden_dim * 2) + dec_hidden_dim, dec_hidden_dim)
        self.v = nn.Linear(dec_hidden_dim, 1, bias=False)

    def forward(self, hidden, enc_outputs, mask=None):
        # hidden: [batch_size, dec_hidden_dim]
        # enc_outputs: [batch_size, src_len, enc_hidden_dim * 2]
        src_len = enc_outputs.shape[1]

        # Repeat decoder hidden state across source sentence length
        hidden_expanded = hidden.unsqueeze(1).repeat(1, src_len, 1)
        energy = torch.tanh(self.attn(torch.cat((hidden_expanded, enc_outputs), dim=2)))
        attention = self.v(energy).squeeze(2)  # [batch_size, src_len]

        if mask is not None:
            attention = attention.masked_fill(mask == 0, -1e10)

        return torch.softmax(attention, dim=1)

# 3. Attention Decoder
class Decoder(nn.Module):
    def __init__(self, vocab_size, embed_dim, enc_hidden_dim, dec_hidden_dim, dropout=0.25, attention=None):
        super().__init__()
        self.vocab_size = vocab_size
        self.attention = attention
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=PAD_IDX)
        # Input to GRU: current word embedding + context vector from attention
        self.rnn = nn.GRU((enc_hidden_dim * 2) + embed_dim, dec_hidden_dim, batch_first=True)
        self.fc_out = nn.Linear((enc_hidden_dim * 2) + dec_hidden_dim + embed_dim, vocab_size)
        self.dropout = nn.Dropout(dropout)

    def forward(self, input_token, hidden, enc_outputs, mask=None):
        # input_token: [batch_size]
        input_token = input_token.unsqueeze(1)
        embedded = self.dropout(self.embedding(input_token))  # [batch_size, 1, embed_dim]

        # Calculate attention weights and context vector
        a = self.attention(hidden, enc_outputs, mask).unsqueeze(1)  # [batch_size, 1, src_len]
        context = torch.bmm(a, enc_outputs)  # [batch_size, 1, enc_hidden_dim * 2]

        rnn_input = torch.cat((embedded, context), dim=2)
        output, hidden = self.rnn(rnn_input, hidden.unsqueeze(0))
        # output: [batch_size, 1, dec_hidden_dim], hidden: [1, batch_size, dec_hidden_dim]

        hidden = hidden.squeeze(0)
        prediction = self.fc_out(torch.cat((output.squeeze(1), context.squeeze(1), embedded.squeeze(1)), dim=1))
        return prediction, hidden, a.squeeze(1)

# 4. Seq2Seq Top-Level Wrapper
class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder, device):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.device = device

    def create_mask(self, src):
        return (src != PAD_IDX)

    def forward(self, src, tgt, teacher_forcing_ratio=0.5):
        batch_size = src.shape[0]
        tgt_len = tgt.shape[1]
        tgt_vocab_size = self.decoder.vocab_size

        outputs = torch.zeros(batch_size, tgt_len, tgt_vocab_size, device=self.device)
        enc_outputs, hidden = self.encoder(src)
        mask = self.create_mask(src)

        # First input token is always BOS (<bos>=1)
        input_token = tgt[:, 0]
        for t in range(1, tgt_len):
            output, hidden, _ = self.decoder(input_token, hidden, enc_outputs, mask)
            outputs[:, t] = output

            # Teacher forcing: feed ground truth with probability teacher_forcing_ratio
            is_teacher_force = random.random() < teacher_forcing_ratio
            top1 = output.argmax(1)
            input_token = tgt[:, t] if is_teacher_force else top1

        return outputs

# Instantiate 8,000-vocabulary model
VOCAB_SIZE = sp.get_piece_size()
EMBED_DIM = 256
ENC_HIDDEN = 512
DEC_HIDDEN = 512

attn = Attention(ENC_HIDDEN, DEC_HIDDEN)
enc = Encoder(VOCAB_SIZE, EMBED_DIM, ENC_HIDDEN, DEC_HIDDEN)
dec = Decoder(VOCAB_SIZE, EMBED_DIM, ENC_HIDDEN, DEC_HIDDEN, attention=attn)

model = Seq2Seq(enc, dec, DEVICE).to(DEVICE)

def count_parameters(m):
    return sum(p.numel() for p in m.parameters() if p.requires_grad)

print(f"Model initialized with {count_parameters(model):,} trainable parameters.")

Using device: cuda
Model initialized with 24,873,280 trainable parameters.


In [9]:
import time
import math
from pathlib import Path

CHECKPOINT_DIR = Path("checkpoints")
CHECKPOINT_DIR.mkdir(exist_ok=True)
BEST_MODEL_PATH = CHECKPOINT_DIR / "best_shona_ndebele_model.pt"

optimizer = torch.optim.Adam(model.parameters(), lr=5e-4)
criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX)

def train_epoch(model, loader, optimizer, criterion, clip=1.0):
    model.train()
    epoch_loss = 0
    for i, (src, tgt) in enumerate(loader):
        src, tgt = src.to(DEVICE), tgt.to(DEVICE)
        optimizer.zero_grad()

        output = model(src, tgt, teacher_forcing_ratio=0.5)
        # Flatten outputs and targets, ignoring the first token (<bos>)
        output_dim = output.shape[-1]
        output = output[:, 1:].reshape(-1, output_dim)
        tgt = tgt[:, 1:].reshape(-1)

        loss = criterion(output, tgt)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), clip)
        optimizer.step()

        epoch_loss += loss.item()
    return epoch_loss / len(loader)

def evaluate(model, loader, criterion):
    model.eval()
    epoch_loss = 0
    with torch.no_grad():
        for src, tgt in loader:
            src, tgt = src.to(DEVICE), tgt.to(DEVICE)
            # Disable teacher forcing entirely during validation
            output = model(src, tgt, teacher_forcing_ratio=0.0)

            output_dim = output.shape[-1]
            output = output[:, 1:].reshape(-1, output_dim)
            tgt = tgt[:, 1:].reshape(-1)

            loss = criterion(output, tgt)
            epoch_loss += loss.item()
    return epoch_loss / len(loader)

# Run 10 training epochs
EPOCHS = 10
best_val_loss = float("inf")

print("Starting training on GPU...")
for epoch in range(1, EPOCHS + 1):
    start_time = time.time()

    train_loss = train_epoch(model, train_loader, optimizer, criterion)
    val_loss = evaluate(model, val_loader, criterion)

    elapsed = time.time() - start_time

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), BEST_MODEL_PATH)
        saved_tag = "[* Saved Best]"
    else:
        saved_tag = ""

    print(f"Epoch: {epoch:02d} | Time: {elapsed:.1f}s | Train Loss: {train_loss:.4f} (PPL: {math.exp(train_loss):.2f}) | Val Loss: {val_loss:.4f} (PPL: {math.exp(val_loss):.2f}) {saved_tag}")

Starting training on GPU...
Epoch: 01 | Time: 584.7s | Train Loss: 6.6532 (PPL: 775.25) | Val Loss: 6.4416 (PPL: 627.41) [* Saved Best]
Epoch: 02 | Time: 597.5s | Train Loss: 5.4856 (PPL: 241.20) | Val Loss: 5.9701 (PPL: 391.56) [* Saved Best]
Epoch: 03 | Time: 594.4s | Train Loss: 4.8239 (PPL: 124.45) | Val Loss: 5.8029 (PPL: 331.25) [* Saved Best]
Epoch: 04 | Time: 594.4s | Train Loss: 4.4110 (PPL: 82.35) | Val Loss: 5.7312 (PPL: 308.33) [* Saved Best]
Epoch: 05 | Time: 595.7s | Train Loss: 4.1121 (PPL: 61.08) | Val Loss: 5.7388 (PPL: 310.70) 
Epoch: 06 | Time: 596.6s | Train Loss: 3.8844 (PPL: 48.64) | Val Loss: 5.7640 (PPL: 318.61) 
Epoch: 07 | Time: 596.5s | Train Loss: 3.7102 (PPL: 40.86) | Val Loss: 5.7502 (PPL: 314.25) 
Epoch: 08 | Time: 595.7s | Train Loss: 3.5534 (PPL: 34.93) | Val Loss: 5.8288 (PPL: 339.97) 
Epoch: 09 | Time: 595.8s | Train Loss: 3.4136 (PPL: 30.37) | Val Loss: 5.8314 (PPL: 340.84) 
Epoch: 10 | Time: 597.1s | Train Loss: 3.2783 (PPL: 26.53) | Val Loss: 5.908

In [10]:
# Load the best saved weights
model.load_state_dict(torch.load(BEST_MODEL_PATH, map_location=DEVICE))
model.eval()

def translate_sentence(sentence: str, target_lang: str = "nd", max_len: int = 64):
    """
    Translates input text.
    target_lang: 'nd' for Ndebele (prepends <2nd>), 'sn' for Shona (prepends <2sn>).
    """
    prefix = "<2nd>" if target_lang == "nd" else "<2sn>"
    formatted_input = f"{prefix} {sentence.strip()}"

    # 1. Encode with BPE
    src_ids = [BOS_IDX] + sp.encode(formatted_input, out_type=int) + [EOS_IDX]
    src_tensor = torch.tensor(src_ids, dtype=torch.long, device=DEVICE).unsqueeze(0)

    with torch.no_grad():
        enc_outputs, hidden = model.encoder(src_tensor)
        mask = model.create_mask(src_tensor)

        # Start decoding with <bos>
        curr_token = torch.tensor([BOS_IDX], dtype=torch.long, device=DEVICE)
        translated_ids = []

        for _ in range(max_len):
            output, hidden, _ = model.decoder(curr_token, hidden, enc_outputs, mask)
            top1 = output.argmax(1).item()

            if top1 == EOS_IDX:
                break
            translated_ids.append(top1)
            curr_token = torch.tensor([top1], dtype=torch.long, device=DEVICE)

    return sp.decode(translated_ids)

# Test translations
test_shona = "Pakutanga Mwari akasika matenga nenyika."
test_ndebele = "Ekuqaleni uNkulunkulu wadala amazulu lomhlaba."

print("--- Test 1: Shona -> Ndebele ---")
print(f"Input:       {test_shona}")
print(f"Translation: {translate_sentence(test_shona, target_lang='nd')}\n")

print("--- Test 2: Ndebele -> Shona ---")
print(f"Input:       {test_ndebele}")
print(f"Translation: {translate_sentence(test_ndebele, target_lang='sn')}")

--- Test 1: Shona -> Ndebele ---
Input:       Pakutanga Mwari akasika matenga nenyika.
Translation: Ekuqaleni uNkulunkulu wa amazulu lomhlaba amazulu lomhlaba.

--- Test 2: Ndebele -> Shona ---
Input:       Ekuqaleni uNkulunkulu wadala amazulu lomhlaba.
Translation: Pakutanga Mwari aka matenga matenga nenyika nenyika.
